# Exploring Translation Mechanism of Large Language Models


# Setup

In [2]:

from copy import deepcopy
import torch
import torch.nn as nn
from tqdm import tqdm
from easy_transformer.EasyTransformer import (
    EasyTransformer,
)
from time import ctime
from functools import partial
import numpy as np

from easy_transformer.experiments import (
    ExperimentMetric,
    AblationConfig,
    EasyAblation,
    EasyPatching,
    PatchingConfig,
)
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import random
import einops
from IPython import get_ipython
from copy import deepcopy
# * custom code here!!!
from translation_dataset import TranslationDataset
from translation_utils import get_translation_logits, show_attention_patterns, get_translation_acc, get_layer_mlp_in, get_layer_mlp_out, batch_path_patching
from collections import defaultdict
# *
from easy_transformer.ioi_utils import (
    path_patching,
    max_2d,
    CLASS_COLORS,
    show_pp,
    show_attention_patterns,
    scatter_attention_and_contribution,
)
from random import randint as ri
from easy_transformer.ioi_circuit_extraction import (
    do_circuit_extraction,
    get_heads_circuit,
    get_mlps_circuit,
    CIRCUIT,
)
from easy_transformer.ioi_utils import logit_diff, probs
from easy_transformer.ioi_utils import get_top_tokens_and_probs as g

ipython = get_ipython()
if ipython is not None:
    ipython.magic("load_ext autoreload")
    ipython.magic("autoreload 2")

Initialise model (use larger N or fewer templates for no warnings about in-template ablation)

In [4]:
model = EasyTransformer.from_pretrained("meta-llama/Llama-2-7b", fold_ln=True, center_writing_weights=False, center_unembed=False)
model.set_use_attn_result(True)
N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads

## Initialise dataset

In [5]:

translation_dataset = TranslationDataset(
    tokenizer=model.tokenizer,
    src_lang='en',
    tgt_lang='zh',
    single_token_only=True,
    multi_token_only=False,
    preprocess_df_trans=None,
    flipped_type='task',
)  # TODO make this a seeded dataset

print(f"Here are two of the prompts from the translation dataset: {translation_dataset.sentences[:2]}")


flipped_translation_dataset = translation_dataset.gen_filpped_dataset()
print(f"Here are two of the prompts from the flipped dataset: {flipped_translation_dataset.sentences[:2]}")

# See logit difference

In [6]:
model.reset_hooks()
model_translation_logits = get_translation_logits(model, translation_dataset, all=True)

model.reset_hooks()
model_flipped_translation_logits = get_translation_logits(model, flipped_translation_dataset, all=True)
print(f'[DEBUG] model_translation_logits mean: {torch.mean(model_translation_logits)}')
print(f'[DEBUG] model_flipped_translation_logits mean: {torch.mean(model_flipped_translation_logits)}')

In [7]:
model.reset_hooks()
model_translation_accuracy = get_translation_acc(model, translation_dataset)
print(f'[DEBUG] model_translation_accuracy mean: {model_translation_accuracy * 100}%')

model.reset_hooks()
model_flipped_translation_accuracy = get_translation_acc(model, flipped_translation_dataset)
print(f'[DEBUG] model_flipped_translation_accuracy mean: {model_flipped_translation_accuracy * 100}%')

# Standard Path patching

In [ ]:
def get_sorted_indices_below_threshold(results, threshold=-0.1):
    # 找到所有符合条件的元素（小于 threshold）
    mask = results < threshold
    # 获取符合条件的元素的行和列索引
    indices = torch.nonzero(mask, as_tuple=True)
    row_indices, col_indices = indices[0], indices[1]
    
    # 获取符合条件的值
    values = results[row_indices, col_indices]
    
    # 按照元素值从小到大排序
    sorted_indices = torch.argsort(values)
    
    # 返回排序后的行列索引
    sorted_row_indices = row_indices[sorted_indices]
    sorted_col_indices = col_indices[sorted_indices]
    
    return torch.stack((sorted_row_indices, sorted_col_indices), dim=1)

def get_top_k_indices(results, k):
    # 获取 top K 元素的值和索引
    values, indices = torch.topk(-results.flatten(), k)
    # 重新将索引转换为二维索引（与原始 tensor 的形状匹配）
    row_indices = indices // results.size(1)  # 计算行索引
    col_indices = indices % results.size(1)  # 计算列索引
    return torch.stack((row_indices, col_indices), dim=1)

In [ ]:
def plot_path_patching(
    model,
    translation_dataset,
    flipped_translation_dataset,
    receiver_hooks,  # list of tuples (hook_name, idx). If idx is not None, then at dim 2 index in with idx (used for doing things for specific attention heads)
    position,
):
    model.reset_hooks()
    default_translation_logits = get_translation_logits(model, translation_dataset)
    results = torch.zeros(size=(N_LAYERS, N_HEADS))
    mlp_results = torch.zeros(size=(N_LAYERS, 1))
    for source_layer in tqdm(range(N_LAYERS)):
        for source_head_idx in [None] + list(range(N_HEADS)):
            model.reset_hooks()
            print(f'[INFO] Patching (source_layer, source_head_idx): {(source_layer, source_head_idx)}')
            cur_translation_logits = batch_path_patching(
                model=model,
                D_new=flipped_translation_dataset,
                D_orig=translation_dataset,
                sender_heads=[(source_layer, source_head_idx)],
                receiver_hooks=receiver_hooks,
                batch_size=128,
                positions=[position],
                freeze_mlps=False,
                have_internal_interactions=False,
            )
            # cur_translation_logits = get_translation_logits(model, translation_dataset)

            if source_head_idx is None:
                mlp_results[source_layer] = cur_translation_logits - default_translation_logits
            else:
                results[source_layer][source_head_idx] = (
                    cur_translation_logits - default_translation_logits
                )

            if source_layer == 1:
                assert not torch.allclose(results, 0.0 * results), results

            if source_layer == N_LAYERS - 1 and source_head_idx == N_HEADS - 1:
                results /= default_translation_logits
                mlp_results /= default_translation_logits

                results *= 100
                mlp_results *= 100

                top_k_heads = get_top_k_indices(results, k=N_LAYERS)
                key_heads = get_sorted_indices_below_threshold(results)

                print(f"Top-{N_LAYERS} Heads:\n", top_k_heads)

                print(f"Several Key Heads (from most important to less important):\n", key_heads)

                # show attention head results
                fig = show_pp(
                    results,
                    title=f"Effect of patching (Heads->Final Residual Stream State) path",
                    return_fig=True,
                    show_fig=False,
                    bartitle="% change in logits",
                )
                fig.show()
                
                # show attention head results
                fig = show_pp(
                    mlp_results,
                    title=f"Effect of patching (MLPs->Final Residual Stream State) path",
                    return_fig=True,
                    show_fig=False,
                    bartitle="% change in logits",
                )
                fig.show()
                
                return top_k_heads, key_heads, results, mlp_results


top_k_heads, key_heads, head_results, mlp_results = plot_path_patching(
    model,
    translation_dataset,
    flipped_translation_dataset,
    receiver_hooks=[(f"blocks.{N_LAYERS-1}.hook_resid_post", None)],
    position="end",
)

# Subspace Path Patching
Use task steering subspaces to patch only task-relevant directions, following the same receiver hook as the standard path patching plot.

In [ ]:
from translation_utils import task_steering_subspace_identification, subspace_intervene_path_patching, subspace_intervene_path_patching_batch
import warnings
warnings.filterwarnings('ignore')

# Subspace path patching mirroring the standard pipeline
def plot_subspace_path_patching(
    model,
    translation_dataset,
    flipped_translation_dataset,
    receiver_hooks,
    position,
):
    model.reset_hooks()
    results = torch.zeros(size=(N_LAYERS, N_HEADS))
    mlp_results = torch.zeros(size=(N_LAYERS, 1))
    for source_layer in tqdm(range(N_LAYERS)):
        for source_head_idx in [None] + list(range(N_HEADS)):
            # print(f'[INFO] Patching (source_layer, source_head_idx): {(source_layer, source_head_idx)}')
            comp = (f"blocks.{source_layer}.attn.hook_result", source_head_idx) if source_head_idx is not None else (f"blocks.{source_layer}.hook_mlp_out", None)
            W_c = task_steering_subspace_identification(
                model,
                translation_dataset,
                flipped_translation_dataset,
                component=comp,
                rank=1,
                position_key=position,
            )
            subspace_dict = {comp: W_c}
            deltas = subspace_intervene_path_patching_batch(
                model,
                translation_dataset,
                flipped_translation_dataset,
                components=[comp],
                subspace_dict=subspace_dict,
                receiver_hooks=receiver_hooks,
                positions=[position],
                epsilon=1e-8,
            )
            delta_val = float(deltas[comp]) * 100.0
            if source_head_idx is None:
                mlp_results[source_layer] = delta_val
            else:
                results[source_layer][source_head_idx] = delta_val

            if source_layer == 1 and source_head_idx == 1:
                assert not torch.allclose(results, 0.0 * results), results

            if source_layer == N_LAYERS - 1 and source_head_idx == N_HEADS - 1:
                top_k_heads = get_top_k_indices(results, k=N_LAYERS)
                key_heads = get_sorted_indices_below_threshold(results)

                print(f"Top-{N_LAYERS} Heads (subspace patching):", top_k_heads)
                print(f"Several Key Heads (most negative first, subspace):", key_heads)

                fig = show_pp(
                    results,
                    title=f"Subspace effect (Heads->Final Residual Stream State)",
                    return_fig=True,
                    show_fig=False,
                    bartitle="% change in logits",
                )
                fig.show()

                fig = show_pp(
                    mlp_results,
                    title=f"Subspace effect (MLPs->Final Residual Stream State)",
                    return_fig=True,
                    show_fig=False,
                    bartitle="% change in logits",
                )
                fig.show()

                return top_k_heads, key_heads, results, mlp_results


subspace_top_k_heads, subspace_key_heads, subspace_head_results, subspace_mlp_results = plot_subspace_path_patching(
    model,
    translation_dataset,
    flipped_translation_dataset,
    receiver_hooks=[(f"blocks.{N_LAYERS-1}.hook_resid_post", None)],
    position="end",
)


# Validation Experiment

## Knockout Key Heads

In [ ]:
TOPK_HEADS = [(31,  8),
        (30, 18),
        (14, 10),
        (12, 17),
        (16, 26),
        (31,  4),
        (16, 28),
        (15,  9),
        (19, 30),
        (15, 17),
        (22, 10),
        ( 9,  1),
        (15, 11),
        (31, 11),
        (19, 16),
        (22,  6),
        (22,  3),
        (10, 23),
        (20, 18),
        ( 9,  7),
        (16,  6),
        ( 4, 19),
        (16, 10),
        (30, 31),
        (17, 18),
        (14, 12),
        (26,  7),
        (30, 26),
        (15, 19),
        (31, 27),
        (14, 14),
        (18, 26)]

model.reset_hooks()
default_translation_logits = get_translation_logits(model, translation_dataset)
default_translation_accuracy = get_translation_acc(model, translation_dataset)
print(f"[INFO] Recall that the initial logit is {default_translation_logits}")
print(f"[INFO] Recall that the initial translation accuracy is {default_translation_accuracy * 100}%")

for i in range(len(TOPK_HEADS)):
    top_name_movers = TOPK_HEADS[:i + 1]
    print(top_name_movers)
    exclude_heads = [(layer, head_idx) for layer in range(N_LAYERS) for head_idx in range(N_HEADS)]
    for head in top_name_movers:
        exclude_heads.remove(head)

    the_extra_hooks = do_circuit_extraction(
        model=model,
        heads_to_remove=get_heads_circuit(
            ioi_dataset=translation_dataset,
            circuit={"name mover": top_name_movers},
        ),
        mlps_to_remove={},
        ioi_dataset=translation_dataset,
        mean_dataset=flipped_translation_dataset,
        return_hooks=True,
        excluded=exclude_heads,
    )
    model.reset_hooks()
    for hook in the_extra_hooks:
        model.add_hook(*hook)
    hooked_translation_logit = get_translation_logits(model, translation_dataset)
    hooked_translation_accuracy = get_translation_acc(model, translation_dataset)
    print(
        f"[INFO] After knocking out {i + 1} key heads, logit is {hooked_translation_logit}"
    )
    print(
        f"[INFO] After knocking out {i + 1} key heads, translation accuracy is {hooked_translation_accuracy * 100}%"
    )

## Knockout random heads

In [ ]:
# 使用集合来避免重复
RANDOM_HEADS = set()
while len(RANDOM_HEADS) < N_LAYERS:  # 假设我们需要生成 N_LAYERS 个随机head
    random_head = (random.randint(0, N_LAYERS - 1), random.randint(0, N_HEADS - 1))
    if random_head not in TOPK_HEADS:
        RANDOM_HEADS.add(random_head)
# 转换为列表
RANDOM_HEADS = list(RANDOM_HEADS)

print(RANDOM_HEADS)

model.reset_hooks()
default_translation_logits = get_translation_logits(model, translation_dataset)
default_translation_accuracy = get_translation_acc(model, translation_dataset)
print(f"[INFO] Recall that the initial logit is {default_translation_logits}")
print(f"[INFO] Recall that the initial translation accuracy is {default_translation_accuracy * 100}%")

for i in range(len(RANDOM_HEADS)):
    top_name_movers = RANDOM_HEADS[:i + 1]
    print(f"[INFO] knockout HEADS: {top_name_movers}")
    exclude_heads = [(layer, head_idx) for layer in range(N_LAYERS) for head_idx in range(N_HEADS)]
    for head in top_name_movers:
        exclude_heads.remove(head)

    the_extra_hooks = do_circuit_extraction(
        model=model,
        heads_to_remove=get_heads_circuit(
            ioi_dataset=translation_dataset,
            circuit={"name mover": top_name_movers},
        ),
        mlps_to_remove={},
        ioi_dataset=translation_dataset,
        mean_dataset=flipped_translation_dataset,
        return_hooks=True,
        excluded=exclude_heads,
    )
    model.reset_hooks()
    for hook in the_extra_hooks:
        model.add_hook(*hook)
    hooked_translation_logit = get_translation_logits(model, translation_dataset)
    hooked_translation_accuracy = get_translation_acc(model, translation_dataset)
    print(
        f"[INFO] After knocking out {i + 1} random heads, logit is {hooked_translation_logit}"
    )
    print(
        f"[INFO] After knocking out {i + 1} random heads, translation accuracy is {hooked_translation_accuracy * 100}%"
    )

# Visualize attention patterns

In [ ]:
model.reset_hooks()
case_study = random.randint(0, len(translation_dataset) - 1)
attn_results = show_attention_patterns(model, TOPK_HEADS, translation_dataset[case_study: case_study + 1])

# MLP Analysis

In [ ]:
unembedding_matrix = model.W_U.T
cosine_similarity = nn.CosineSimilarity(dim=0, eps=1e-6)

INDICATORS = model.tokenizer.encode("中文 English :", add_special_tokens=False)

mlp_in_results = defaultdict(list)
for i in range(N_LAYERS):
    mlp_in = get_layer_mlp_in(model, translation_dataset, i)
    cos_sim = 0
    random_cos_sim = 0
    indicator_cos_sim = 0
    for j in range(len(translation_dataset)):
        src_token_embedding = torch.zeros(len(translation_dataset.out_tok_ids[j]), mlp_in.shape[-1]).to(mlp_in.device)
        for idx, src_token in enumerate(translation_dataset.out_tok_ids[j]):
            src_token_embedding[idx] = unembedding_matrix[src_token]
        # src_token_embedding /= len(translation_dataset.word_idx["src"][j])
        cos_sim += max([cosine_similarity(mlp_in[j, translation_dataset.word_idx["end"][j]], src_token_embedding[idx]).item() for idx in range(src_token_embedding.shape[0])])
        # take random one
        random_example = j
        while random_example == j:
            random_example = random.randint(0, len(translation_dataset) - 1)
        random_cos_sim += max([cosine_similarity(mlp_in[random_example, translation_dataset.word_idx["end"][random_example]], src_token_embedding[idx]).item() for idx in range(src_token_embedding.shape[0])])
        # Translation Indicator
        indicator_token_embedding = torch.zeros(len(INDICATORS), mlp_in.shape[-1]).to(mlp_in.device)
        for idx, indicator_token in enumerate(INDICATORS):
            indicator_token_embedding[idx] = unembedding_matrix[indicator_token]
        indicator_cos_sim += sum([cosine_similarity(mlp_in[j, translation_dataset.word_idx["end"][j]], indicator_token_embedding[idx]).item() for idx in range(indicator_token_embedding.shape[0])])

    cos_sim /= len(translation_dataset)
    random_cos_sim /= len(translation_dataset)
    indicator_cos_sim /= len(translation_dataset)
    print(f'[INFO] {i} layer <MLP_in, SRC_TOKEN>: {cos_sim}')
    print(f'[INFO] {i} layer <MLP_in, RANDOM_ENG_TOKEN>: {random_cos_sim}')
    print(f'[INFO] {i} layer <MLP_in, INDICATOR_TOKEN>: {indicator_cos_sim}')
    mlp_in_results['src_token'].append(cos_sim)
    mlp_in_results['random_english_token'].append(random_cos_sim)
    mlp_in_results['indicator_token'].append(indicator_cos_sim)

In [ ]:
unembedding_matrix = model.W_U.T
cosine_similarity = nn.CosineSimilarity(dim=0, eps=1e-6)

mlp_out_results = defaultdict(list)
for i in range(N_LAYERS):
    mlp_in = get_layer_mlp_in(model, translation_dataset, i)
    mlp_out = get_layer_mlp_out(model, translation_dataset, i)
    mlp_out = mlp_in - mlp_out 
    cos_sim = 0
    random_cos_sim = 0
    for j in range(len(translation_dataset)):
        tgt_token_embedding = torch.zeros(len(translation_dataset.out_tok_ids[j]), mlp_out.shape[-1]).to(mlp_out.device)
        for idx, tgt_token in enumerate(translation_dataset.out_tok_ids[j]):
            tgt_token_embedding[idx] = unembedding_matrix[tgt_token]
        cos_sim += sum([cosine_similarity(mlp_out[j, translation_dataset.word_idx["end"][j]], tgt_token_embedding[idx]).item() for idx in range(tgt_token_embedding.shape[0])])
        # take random one
        random_example = j
        while random_example == j:
            random_example = random.randint(0, len(translation_dataset) - 1)
        random_cos_sim += max([cosine_similarity(mlp_out[random_example, translation_dataset.word_idx["end"][random_example]], tgt_token_embedding[idx]).item() for idx in range(tgt_token_embedding.shape[0])])
    cos_sim /= len(translation_dataset)
    random_cos_sim /= len(translation_dataset)
    print(f'[INFO] {i} layer <MLP_out-MLP_in, TGT_TOKEN>: {cos_sim}')
    print(f'[INFO] {i} layer <MLP_out-MLP_in, RANDOM_TGT_TOKEN>: {random_cos_sim}')
    mlp_out_results['tgt_token'].append(cos_sim)
    mlp_out_results['random_tgt_token'].append(random_cos_sim)
